# Northstar Incremental Employee Merge

This notebook demonstrates an incremental processing pattern for employee data using a persisted watermark and Delta Lake MERGE semantics.

The workflow simulates a subsequent source arrival containing both new and changed employee records, processes only records newer than the stored watermark, and applies inserts and updates to the Silver employee Delta table without rebuilding the full dataset.

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
# Northstar incremental processing configuration

silver_employee_path = "/Volumes/northstar_dev/silver/workforce/employees"
watermark_path = "/Volumes/northstar_dev/silver/control/employee_watermark"

print(f"Silver employee target: {silver_employee_path}")
print(f"Watermark location: {watermark_path}")

In [ ]:
# Load the persisted watermark.
# If this is the first incremental run, start from the baseline timestamp.

default_watermark = "1900-01-01 00:00:00"

try:
    watermark_df = spark.read.format("delta").load(watermark_path)

    last_watermark = (
        watermark_df
        .agg(F.max("watermark_timestamp").alias("last_watermark"))
        .first()["last_watermark"]
    )

    if last_watermark is None:
        last_watermark = default_watermark

except Exception:
    last_watermark = default_watermark

print(f"Current watermark: {last_watermark}")

In [ ]:
# Simulate a subsequent employee source arrival.
# Existing employee IDs represent updates; new IDs represent inserts.

incremental_data = [
    (1001, "Avery", "Johnson", "Information Technology", "NY", "Active", "2026-08-18 14:00:00"),
    (1002, "Jordan", "Williams", "Finance", "MD", "Active", "2026-08-18 14:05:00"),
    (10001, "Taylor", "Morgan", "Customer Operations", "VA", "Active", "2026-08-18 14:10:00"),
    (10002, "Cameron", "Davis", "Human Resources", "NC", "Active", "2026-08-18 14:15:00")
]

incremental_columns = [
    "employee_id",
    "first_name",
    "last_name",
    "department",
    "state",
    "employment_status",
    "source_updated_at"
]

incoming_df = (
    spark.createDataFrame(incremental_data, incremental_columns)
    .withColumn("source_updated_at", F.to_timestamp("source_updated_at"))
)

incoming_df.show(truncate=False)

In [ ]:
# Filter the incoming batch to records newer than the persisted watermark.

incremental_df = incoming_df.filter(
    F.col("source_updated_at") > F.to_timestamp(F.lit(str(last_watermark)))
)

incremental_count = incremental_df.count()

print(f"Records eligible for incremental processing: {incremental_count}")

incremental_df.orderBy("source_updated_at").show(truncate=False)

In [ ]:
# Apply incremental changes to the Silver employee Delta table.

silver_delta = DeltaTable.forPath(spark, silver_employee_path)

(
    silver_delta.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.employee_id = source.employee_id"
    )
    .whenMatchedUpdate(set={
        "first_name": "source.first_name",
        "last_name": "source.last_name",
        "department": "source.department",
        "state": "source.state",
        "employment_status": "source.employment_status",
        "source_updated_at": "source.source_updated_at"
    })
    .whenNotMatchedInsert(values={
        "employee_id": "source.employee_id",
        "first_name": "source.first_name",
        "last_name": "source.last_name",
        "department": "source.department",
        "state": "source.state",
        "employment_status": "source.employment_status",
        "source_updated_at": "source.source_updated_at"
    })
    .execute()
)

print("Incremental MERGE completed.")

In [ ]:
# Advance and persist the watermark after a successful MERGE.

new_watermark = (
    incremental_df
    .agg(F.max("source_updated_at").alias("watermark_timestamp"))
    .first()["watermark_timestamp"]
)

if new_watermark is not None:
    watermark_output_df = spark.createDataFrame(
        [(new_watermark,)],
        ["watermark_timestamp"]
    )

    (
        watermark_output_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(watermark_path)
    )

    print(f"Watermark advanced to: {new_watermark}")
else:
    print("No incremental records processed; watermark unchanged.")

In [ ]:
# Validate the incremental result.

updated_silver_df = spark.read.format("delta").load(silver_employee_path)

print(f"Silver employee row count after MERGE: {updated_silver_df.count()}")

updated_silver_df.filter(
    F.col("employee_id").isin(1001, 1002, 10001, 10002)
).orderBy("employee_id").show(truncate=False)